In [ ]:
import tkinter as tk
from tkinter import messagebox
import pymysql

class DictionaryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Python 수업 용어 사전")
        self.root.geometry("450x720")
        
        # 윈도우 기본 시스템 폰트(맑은 고딕) 설정
        self.font_main = ("Malgun Gothic", 10)
        self.font_bold = ("Malgun Gothic", 11, "bold")
        self.font_title = ("Malgun Gothic", 15, "bold")
        
        self.db_host = "127.0.0.1"
        self.connection = None
        
        # 검색 히스토리 관리용
        self.history = []
        self.history_index = -1
        
        # 1. 로그인 화면
        self.login_frame = tk.Frame(self.root)
        self.login_frame.pack(pady=50)
        
        tk.Label(self.login_frame, text="🐍 Python 수업 용어 사전", font=self.font_title).pack(pady=20)
        
        tk.Label(self.login_frame, text="DB 계정 ID:", font=self.font_main).pack()
        self.ent_user = tk.Entry(self.login_frame, font=self.font_main)
        self.ent_user.pack(pady=5)
        self.ent_user.bind('<Return>', lambda event: self.ent_pw.focus())
        
        tk.Label(self.login_frame, text="Password:", font=self.font_main).pack()
        self.ent_pw = tk.Entry(self.login_frame, show="*", font=self.font_main)
        self.ent_pw.pack(pady=5)
        self.ent_pw.bind('<Return>', lambda event: self.connect_db())
        
        tk.Button(self.login_frame, text="데이터베이스 접속", font=self.font_bold,
                  command=self.connect_db, bg="#2196F3", fg="white", width=20).pack(pady=20)

    def connect_db(self):
        try:
            self.connection = pymysql.connect(
                host=self.db_host,
                user=self.ent_user.get(),
                password=self.ent_pw.get(),
                db='class_dict_db',
                charset='utf8mb4',
                cursorclass=pymysql.cursors.DictCursor,
                autocommit=True
            )
            messagebox.showinfo("성공", "서버에 연결되었습니다.")
            self.show_search_ui()
        except Exception as e:
            messagebox.showerror("접속 오류", f"로그인 실패: {e}")

    def show_search_ui(self):
        self.login_frame.destroy()
        
        self.search_frame = tk.Frame(self.root)
        self.search_frame.pack(pady=20, fill="both", expand=True)
        
        tk.Label(self.search_frame, text="검색할 용어를 입력하세요", font=self.font_bold).pack(pady=10)
        
        # 검색창
        self.ent_search = tk.Entry(self.search_frame, width=35, font=self.font_main)
        self.ent_search.pack(pady=5)
        self.ent_search.focus_set()
        
        # 추천 목록 리스트박스
        self.list_suggestion = tk.Listbox(self.search_frame, width=35, height=0, font=self.font_main)
        self.list_suggestion.pack()
        self.list_suggestion.pack_forget() 
        
        self.ent_search.bind('<KeyRelease>', self.on_key_release)
        self.ent_search.bind('<Return>', lambda e: self.search_word())
        self.list_suggestion.bind("<<ListboxSelect>>", self.select_from_list)
        
        # 버튼 프레임 (검색 / 이전 검색)
        btn_frame = tk.Frame(self.search_frame)
        btn_frame.pack(pady=5)
        
        tk.Button(btn_frame, text="단어 검색", font=self.font_bold, command=self.search_word, width=10).pack(side="left", padx=5)
        tk.Button(btn_frame, text="이전 검색", font=self.font_main, command=self.go_back_history, bg="#f9f9f9", width=10).pack(side="left", padx=5)
        
        # 결과창
        self.txt_result = tk.Text(self.search_frame, height=12, width=50, font=self.font_main, padx=10, pady=10)
        self.txt_result.pack(pady=10, padx=20)

        # 새 단어 등록 버튼
        tk.Button(self.search_frame, text="+ 새 용어 등록하기", font=self.font_main,
                  command=self.open_add_window, bg="#eeeeee").pack(pady=10)

    def on_key_release(self, event):
        if event.keysym == "Down" and self.list_suggestion.winfo_viewable():
            self.list_suggestion.focus_set()
            self.list_suggestion.selection_set(0)
            return

        search_text = self.ent_search.get().strip()
        if not search_text:
            self.list_suggestion.pack_forget()
            return

        try:
            self.connection.ping(reconnect=True)
            with self.connection.cursor() as cursor:
                sql = "SELECT word FROM terms WHERE word LIKE %s LIMIT 5"
                cursor.execute(sql, (f"%{search_text}%",))
                results = cursor.fetchall()
                
                if results:
                    self.list_suggestion.delete(0, tk.END)
                    for item in results:
                        self.list_suggestion.insert(tk.END, item['word'])
                    self.list_suggestion.config(height=len(results))
                    self.list_suggestion.pack(after=self.ent_search)
                else:
                    self.list_suggestion.pack_forget()
        except:
            self.list_suggestion.pack_forget()

    def select_from_list(self, event):
        if not self.list_suggestion.curselection(): return
        selected_word = self.list_suggestion.get(self.list_suggestion.curselection())
        self.ent_search.delete(0, tk.END)
        self.ent_search.insert(0, selected_word)
        self.list_suggestion.pack_forget()
        self.search_word()

    def search_word(self):
        word = self.ent_search.get().strip()
        self.list_suggestion.pack_forget()
        if not word: return
        
        try:
            with self.connection.cursor() as cursor:
                sql = "SELECT definition FROM terms WHERE word = %s"
                cursor.execute(sql, (word,))
                result = cursor.fetchone()
                
                self.txt_result.delete(1.0, tk.END)
                if result:
                    self.txt_result.insert(tk.END, f"[{word}]\n\n{result['definition']}")
                    
                    # 검색 히스토리 저장 (중복 저장 방지 및 최대 30개 유지)
                    if not self.history or self.history[-1] != word:
                        self.history.append(word)
                        if len(self.history) > 30:
                            self.history.pop(0)
                    self.history_index = len(self.history) - 1
                else:
                    self.txt_result.insert(tk.END, "검색 결과가 없습니다.")
        except Exception as e:
            messagebox.showerror("오류", f"조회 중 오류 발생: {e}")

    def go_back_history(self):
        """이전 검색 단어로 이동"""
        if len(self.history) <= 1 or self.history_index <= 0:
            messagebox.showinfo("알림", "이전 검색 기록이 없습니다.")
            return
        
        self.history_index -= 1
        prev_word = self.history[self.history_index]
        
        self.ent_search.delete(0, tk.END)
        self.ent_search.insert(0, prev_word)
        self.search_word() # 해당 단어로 재검색 실행

    def open_add_window(self):
        self.add_win = tk.Toplevel(self.root)
        self.add_win.title("새 용어 등록")
        self.add_win.geometry("380x480")
        
        tk.Label(self.add_win, text="📖 새 용어 추가", font=self.font_bold).pack(pady=15)
        
        tk.Label(self.add_win, text="단어명:", font=self.font_main).pack()
        self.ent_new_word = tk.Entry(self.add_win, width=30, font=self.font_main)
        self.ent_new_word.pack(pady=5)
        self.ent_new_word.focus_set()

        tk.Label(self.add_win, text="단어 내용(뜻):", font=self.font_main).pack(pady=5)
        self.txt_new_def = tk.Text(self.add_win, height=10, width=40, font=self.font_main, padx=5, pady=5)
        self.txt_new_def.pack(pady=5)
        
        tk.Button(self.add_win, text="DB 기록 완료", font=self.font_bold,
                  command=self.save_to_db, bg="#4CAF50", fg="white", width=20).pack(pady=20)

    def save_to_db(self):
        new_word = self.ent_new_word.get().strip()
        new_def = self.txt_new_def.get(1.0, tk.END).strip()
        
        if not new_word or not new_def:
            messagebox.showwarning("누락", "모든 항목을 입력하세요.")
            return

        try:
            with self.connection.cursor() as cursor:
                # 중복 체크
                check_sql = "SELECT word FROM terms WHERE word = %s"
                cursor.execute(check_sql, (new_word,))
                if cursor.fetchone():
                    messagebox.showwarning("중복 오류", f"'{new_word}'은(는) 이미 등록된 단어입니다.")
                    return

                sql = "INSERT INTO terms (word, definition) VALUES (%s, %s)"
                cursor.execute(sql, (new_word, new_def))
                self.connection.commit()
                
            messagebox.showinfo("성공", f"'{new_word}' 등록 성공!")
            self.add_win.destroy()
        except Exception as e:
            messagebox.showerror("저장 오류", f"DB 기록 실패: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = DictionaryApp(root)
    root.mainloop()